In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import make_classification, make_regression
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, mean_squared_error, r2_score, classification_report
import warnings
warnings.filterwarnings('ignore')

# 시드 고정 (재현성)
np.random.seed(42)

In [ ]:
X_clf, y_clf = make_classification(
    n_samples=400, n_features=2, random_state=42,
    n_informative=2, n_redundant=0,n_clusters_per_class=1,class_sep=1.5
    )
scaler = StandardScaler()
# 데이터 분할 train - test 8:2
x_train,x_test,y_train,y_test = train_test_split(X_clf,y_clf,random_state=42, stratify=y_clf,test_size=0.2)
x_train = scaler.fit_transform(x_train)
x_test = scaler.transform(x_test)
# pytorch 텐서로 변환
x_train_t = torch.tensor(x_train,dtype=torch.float32)
y_train_t = torch.FloatTensor(y_train)
x_test_t = torch.FloatTensor(x_test)
y_test_t = torch.FloatTensor(y_test)

In [ ]:
# 분류모델 정의
class BinaryClassifier(nn.Module):
  def __init__(self) -> None:
    super().__init__()
    self.fc1 = nn.Linear( 2 , 16)
    self.relu = nn.ReLU()
    self.fc2 = nn.Linear(16 ,1 )    
  def forward(self, x):    
    x = self.relu( self.fc1(x) )    
    output = self.fc2(x) 
    return output
model = BinaryClassifier()    
model

In [ ]:
criterion = nn.BCEWithLogitsLoss()
optimizer = optim.SGD(model.parameters(), lr=0.01,momentum=0)
x_train_dataset = TensorDataset(x_train_t, y_train_t)
x_train_loader = DataLoader(x_train_dataset,batch_size=1,shuffle=True)

In [ ]:
# 훈련루프
from tqdm import tqdm
epochs = 100
train_losses , train_accs = [], []
for epoch in tqdm(range(epochs)):
  total_loss, total_acc = 0.0 , 0.0
  for x,y in x_train_loader:
    # 가중치 초기화
    optimizer.zero_grad()
    # forward
    predict = model(x).squeeze(1)
    # loss 구하고        
    loss = criterion(predict,y)
    # backward
    loss.backward()
    # 가중치 업데이트
    optimizer.step()

    total_loss += loss.item()
    total_acc += sum(predict == y) / len(y)
  train_losses.append( total_loss / len(x_train_loader) )
  train_accs.append( total_acc / len(x_train_loader) )
